# Synthesize a `dev_system`-style dataset from the real `system.*` tables

Produces a non-trivial Databricks cost dataset suitable for developing the cloud-infra-costs FOCUS query and other cost analysis without needing access to the real production `system.*` schemas.

**Hybrid approach (per ADR-0010):**
- **Real rows kept as templates** — `system.billing.usage` has 11 real rows from the AI-pricing test workload covering MODEL_SERVING, AI_GATEWAY, SQL, JOBS
- **Real freebies copied verbatim** — `system.billing.list_prices` (946 rows, full SKU catalog), `workspaces_latest`, `clusters`, `warehouses`
- **Perturbation** — extra synthetic rows generated across 90 days × N workspaces × N items to give realistic volume and distribution
- **Invention** — categories missing from real data (DLT, VECTOR_SEARCH, LAKEHOUSE_MONITORING, PREDICTIVE_OPTIMIZATION, ONLINE_TABLES, AI_FUNCTIONS, DATABASE, NETWORKING, AGENT_EVALUATION, ALL_PURPOSE, INTERACTIVE, NOTEBOOKS, SHARED_SERVERLESS_COMPUTE) are generated using realistic SKU names drawn from `list_prices`
- **Tags** — Service_ID hierarchy with deliberate hygiene gaps (70% well-tagged, 20% malformed, 10% untagged) so the FOCUS pipeline's Untagged-bucket logic gets exercised
- **`account_prices`** — fabricated as ~85% of `list_prices` to simulate a contracted discount delta (the real `account_prices` table is absent on accounts without negotiated pricing)

**Catalogue layout**: tables go under the workspace's default catalogue in four `dev_*` schemas — `<catalog>.dev_billing.*`, `<catalog>.dev_compute.*`, `<catalog>.dev_access.*`, `<catalog>.dev_lakeflow.*`. (Creating a separate top-level catalogue fails on Default-Storage workspaces without a configured metastore root.)

**Validated 2026-06-13**: cells 1–4 (catalogue setup and freebie copy) verified working via SQL Statements API against `dbx_aiprice_mw929` catalogue in the test workspace. Cells 5–7 (synthetic dimensions and fact rows via PySpark) need a notebook environment to run.

In [ ]:
# ---- Parameters ----
DEV_CATALOG = "dbx_aiprice_mw929"     # workspace's default UC catalogue (created by the workspace)
BILLING = f"{DEV_CATALOG}.dev_billing"
COMPUTE = f"{DEV_CATALOG}.dev_compute"
ACCESS = f"{DEV_CATALOG}.dev_access"
LAKEFLOW = f"{DEV_CATALOG}.dev_lakeflow"

DAYS = 90
N_WORKSPACES = 3
ITEMS_PER_WORKSPACE = (8, 18)
OPS_PER_ITEM_PER_DAY = (1, 4)
SEED = 42

# Tag-hygiene mix (sum should be 1.0)
TAG_MIX = {"well_formed": 0.7, "malformed": 0.2, "untagged": 0.1}

# Account-price discount vs list (15%)
ACCOUNT_DISCOUNT = 0.85

from datetime import date, datetime, timedelta, timezone
import random
import uuid

random.seed(SEED)
TODAY = date.today()
START_DATE = TODAY - timedelta(days=DAYS)

## 1. Create schemas under the workspace's default catalogue

Avoids the metastore-root-URL issue with `CREATE CATALOG` on Default-Storage workspaces. If the FOCUS query needs to be retargeted from `system.*`, the find-and-replace is:

- `system.billing.` → `<DEV_CATALOG>.dev_billing.`
- `system.access.`  → `<DEV_CATALOG>.dev_access.`
- `system.compute.` → `<DEV_CATALOG>.dev_compute.`
- `system.lakeflow.` → `<DEV_CATALOG>.dev_lakeflow.`

In [ ]:
for schema in (BILLING, COMPUTE, ACCESS, LAKEFLOW):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
print("Schemas ready")

## 2. Clone schemas from `system.*` and copy real data

Uses `CREATE TABLE ... USING DELTA AS SELECT * FROM system.X WHERE 1=0` to clone column types (including the deeply nested `usage_metadata` struct) without taking any rows. Real rows are then inserted in the next step. `CREATE TABLE LIKE system.*` fails because system tables don't expose their datasource cleanly.

In [ ]:
CLONES = [
    (f"{BILLING}.usage",               "system.billing.usage"),
    (f"{BILLING}.list_prices",         "system.billing.list_prices"),
    (f"{ACCESS}.workspaces_latest",    "system.access.workspaces_latest"),
    (f"{COMPUTE}.clusters",            "system.compute.clusters"),
    (f"{COMPUTE}.warehouses",          "system.compute.warehouses"),
    (f"{LAKEFLOW}.pipelines",          "system.lakeflow.pipelines"),
]
for target, source in CLONES:
    spark.sql(f"DROP TABLE IF EXISTS {target}")
    spark.sql(f"CREATE TABLE {target} USING DELTA AS SELECT * FROM {source} WHERE 1=0")

# Copy the freebies (small, authoritative)
spark.sql(f"INSERT INTO {BILLING}.list_prices SELECT * FROM system.billing.list_prices")
spark.sql(f"INSERT INTO {ACCESS}.workspaces_latest SELECT * FROM system.access.workspaces_latest")
spark.sql(f"INSERT INTO {COMPUTE}.clusters SELECT * FROM system.compute.clusters")
spark.sql(f"INSERT INTO {COMPUTE}.warehouses SELECT * FROM system.compute.warehouses")
spark.sql(f"INSERT INTO {BILLING}.usage SELECT * FROM system.billing.usage")
print("Schemas cloned and freebies copied")

## 3. Fabricate `account_prices`

The real `system.billing.account_prices` table is **absent** on accounts without negotiated pricing — it's not just empty, it doesn't exist. We mirror `list_prices`' shape and discount by 15% so the FOCUS query's contracted-cost join produces a non-trivial delta vs list.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {BILLING}.account_prices")
spark.sql(f"""
CREATE TABLE {BILLING}.account_prices USING DELTA AS
SELECT
    account_id, price_start_time, price_end_time, sku_name, cloud, currency_code, usage_unit,
    named_struct(
        'default', pricing.default * {ACCOUNT_DISCOUNT},
        'promotional', pricing.promotional,
        'effective_list', pricing.effective_list
    ) AS pricing
FROM system.billing.list_prices
""")
n = spark.sql(f"SELECT COUNT(*) FROM {BILLING}.account_prices").collect()[0][0]
print(f"account_prices: {n} rows at {int(ACCOUNT_DISCOUNT*100)}% of list")

## 4. Build synthetic dimensions

Synthetic workspace IDs, items, clusters, warehouses, pipelines — generated up-front so the fact table can reference consistent IDs across rows. The real workspace/cluster/warehouse rows already loaded above are kept; synthetics layer on top with distinct IDs.

In [ ]:
ACCOUNT_ID = spark.sql("SELECT account_id FROM system.access.workspaces_latest LIMIT 1").collect()[0][0]
REAL_WORKSPACE_ID = spark.sql("SELECT workspace_id FROM system.access.workspaces_latest LIMIT 1").collect()[0][0]

SYNTH_WORKSPACES = [
    {"workspace_id": str(random.randint(10**15, 10**16 - 1)),
     "workspace_name": f"synth-ws-{i:02d}"}
    for i in range(N_WORKSPACES)
]
ALL_WORKSPACES = [{"workspace_id": str(REAL_WORKSPACE_ID), "workspace_name": "real"}] + SYNTH_WORKSPACES

ITEMS, CLUSTERS, WAREHOUSES, PIPELINES = {}, [], [], []

DIVISIONS = ["Research", "Commercial", "Platform", "Finance"]
PLATFORMS = ["Data", "Customer", "Internal", "ML"]
WORKLOADS = ["Analytics", "Ingest", "ML", "Reporting", "ETL"]
COMPONENTS = ["Bronze", "Silver", "Gold", "Streaming", "Batch"]

def gen_service_id():
    r = random.random()
    if r < TAG_MIX["well_formed"]:
        return "/".join([random.choice(DIVISIONS), random.choice(PLATFORMS),
                         random.choice(WORKLOADS), random.choice(COMPONENTS)])
    if r < TAG_MIX["well_formed"] + TAG_MIX["malformed"]:
        return random.choice(["", "BadlyTagged", "Research//Workload", "Only/Two"])
    return None  # Service_ID key absent

def gen_tags():
    sid = gen_service_id()
    tags = {"Environment": random.choice(["prod", "dev", "test", "staging"])}
    if sid is not None:
        tags["Service_ID"] = sid
    return tags

ITEM_CATEGORIES = ["job", "cluster", "warehouse", "endpoint", "pipeline",
                   "vector_index", "notebook", "online_table", "database_instance"]

for ws in ALL_WORKSPACES:
    items = []
    for _ in range(random.randint(*ITEMS_PER_WORKSPACE)):
        cat = random.choice(ITEM_CATEGORIES)
        item = {"workspace_id": ws["workspace_id"], "category": cat, "tags": gen_tags()}
        if cat == "job":
            item["job_id"] = str(random.randint(10**14, 10**15 - 1))
        elif cat == "cluster":
            cid = f"{random.randint(1000,9999):04d}-{random.randint(100000,999999)}-{uuid.uuid4().hex[:8]}"
            item["cluster_id"] = cid
            CLUSTERS.append({"workspace_id": ws["workspace_id"], "cluster_id": cid,
                             "cluster_name": f"synth-cl-{cid[-8:]}", "tags": item["tags"]})
        elif cat == "warehouse":
            wid = uuid.uuid4().hex[:16]
            item["warehouse_id"] = wid
            WAREHOUSES.append({"workspace_id": ws["workspace_id"], "warehouse_id": wid,
                               "warehouse_name": f"synth-wh-{wid[:8]}", "tags": item["tags"]})
        elif cat == "endpoint":
            item["endpoint_id"] = uuid.uuid4().hex[:24]
        elif cat == "pipeline":
            pid = uuid.uuid4().hex
            item["dlt_pipeline_id"] = pid
            PIPELINES.append({"workspace_id": ws["workspace_id"], "pipeline_id": pid,
                              "name": f"synth-pl-{pid[:8]}", "tags": item["tags"]})
        elif cat == "database_instance":
            item["database_instance_id"] = uuid.uuid4().hex[:24]
        items.append(item)
    ITEMS[ws["workspace_id"]] = items

total_items = sum(len(v) for v in ITEMS.values())
print(f"Generated {len(SYNTH_WORKSPACES)} synthetic workspaces, {total_items} items, "
      f"{len(CLUSTERS)} clusters, {len(WAREHOUSES)} warehouses, {len(PIPELINES)} pipelines")

## 5. Insert synthetic dimension rows

Schema-driven: pull a real template row from each table and only override the identifying fields, leaving complex nested types (cluster `tags` struct etc.) in their original shape.

In [ ]:
from pyspark.sql import Row
now_ts = datetime.now(timezone.utc)

# Explicit schemas: rows we build can have all-None columns where PySpark's
# inference fails (CANNOT_DETERMINE_TYPE). Pass schema explicitly to fix.
SCHEMAS = {
    "workspaces_latest": spark.table(f"{ACCESS}.workspaces_latest").schema,
    "clusters":          spark.table(f"{COMPUTE}.clusters").schema,
    "warehouses":        spark.table(f"{COMPUTE}.warehouses").schema,
    "pipelines":         spark.table(f"{LAKEFLOW}.pipelines").schema,
}

def cols_of(name):
    return [f.name for f in SCHEMAS[name].fields]

def template_row(table, name):
    """Real row (if any) to inherit nested-struct shape; otherwise all-None dict."""
    rows = spark.sql(f"SELECT * FROM {table} LIMIT 1").collect()
    return rows[0].asDict() if rows else {c: None for c in cols_of(name)}

# Workspaces
if SYNTH_WORKSPACES:
    t = template_row(f"{ACCESS}.workspaces_latest", "workspaces_latest")
    rows = []
    for w in SYNTH_WORKSPACES:
        r = dict(t)
        r["workspace_id"] = w["workspace_id"]
        r["workspace_name"] = w["workspace_name"]
        if "workspace_url" in r:
            r["workspace_url"] = f"https://adb-{w['workspace_id']}.0.azuredatabricks.net"
        rows.append(tuple(r[c] for c in cols_of("workspaces_latest")))
    spark.createDataFrame(rows, schema=SCHEMAS["workspaces_latest"]).write.mode("append").saveAsTable(f"{ACCESS}.workspaces_latest")
    print(f"+ {len(rows)} synth workspaces")

# Clusters
if CLUSTERS:
    t = template_row(f"{COMPUTE}.clusters", "clusters")
    rows = []
    for c in CLUSTERS:
        r = dict(t)
        r["account_id"] = ACCOUNT_ID
        r["workspace_id"] = c["workspace_id"]
        r["cluster_id"] = c["cluster_id"]
        r["cluster_name"] = c["cluster_name"]
        r["tags"] = c["tags"]
        r["change_time"] = now_ts
        rows.append(tuple(r[col] for col in cols_of("clusters")))
    spark.createDataFrame(rows, schema=SCHEMAS["clusters"]).write.mode("append").saveAsTable(f"{COMPUTE}.clusters")
    print(f"+ {len(rows)} synth clusters")

# Warehouses
if WAREHOUSES:
    t = template_row(f"{COMPUTE}.warehouses", "warehouses")
    rows = []
    for w in WAREHOUSES:
        r = dict(t)
        r["account_id"] = ACCOUNT_ID
        r["workspace_id"] = w["workspace_id"]
        r["warehouse_id"] = w["warehouse_id"]
        r["warehouse_name"] = w["warehouse_name"]
        r["tags"] = w["tags"]
        r["change_time"] = now_ts
        rows.append(tuple(r[col] for col in cols_of("warehouses")))
    spark.createDataFrame(rows, schema=SCHEMAS["warehouses"]).write.mode("append").saveAsTable(f"{COMPUTE}.warehouses")
    print(f"+ {len(rows)} synth warehouses")

# Pipelines (system.lakeflow.pipelines has zero real rows, so we build from schema)
if PIPELINES:
    rows = []
    for p in PIPELINES:
        r = {col: None for col in cols_of("pipelines")}
        r["account_id"] = ACCOUNT_ID
        r["workspace_id"] = p["workspace_id"]
        r["pipeline_id"] = p["pipeline_id"]
        for opt, val in [("name", p["name"]), ("pipeline_type", "WORKSPACE"),
                          ("tags", p["tags"]), ("change_time", now_ts), ("delete_time", None)]:
            if opt in r:
                r[opt] = val
        rows.append(tuple(r[col] for col in cols_of("pipelines")))
    spark.createDataFrame(rows, schema=SCHEMAS["pipelines"]).write.mode("append").saveAsTable(f"{LAKEFLOW}.pipelines")
    print(f"+ {len(rows)} synth pipelines")

## 6. Generate the usage fact rows

For each item × day, generate 1–N usage rows mapped to the right `billing_origin_product` and SKU. SKU names are drawn from `list_prices` filtered by a regex matching the category — guarantees the SKU exists in the price table (no broken joins) and the FOCUS query's SKU-name branches still match.

In [ ]:
from decimal import Decimal
from pyspark.sql.types import StructType, DecimalType

skus = spark.sql("SELECT sku_name, usage_unit FROM system.billing.list_prices").collect()
DBU_SKUS = [r["sku_name"] for r in skus if r["usage_unit"] == "DBU"]

def pick_sku(category):
    patterns = {
        "JOBS": ["JOBS"],
        "SQL": ["SQL"],
        "MODEL_SERVING": ["INFERENCE", "SERVING"],
        "AI_GATEWAY": ["INFERENCE", "GATEWAY"],
        "AI_FUNCTIONS": ["INFERENCE"],
        "VECTOR_SEARCH": ["VECTOR_SEARCH", "SERVING"],
        "DLT": ["DLT"],
        "LAKEFLOW_CONNECT": ["DLT", "LAKEFLOW"],
        "ONLINE_TABLES": ["ONLINE", "DLT"],
        "LAKEHOUSE_MONITORING": ["JOBS_SERVERLESS"],
        "PREDICTIVE_OPTIMIZATION": ["JOBS_SERVERLESS"],
        "DATABASE": ["JOBS_SERVERLESS", "DATABASE"],
        "NETWORKING": ["NETWORKING"],
        "AGENT_EVALUATION": ["INFERENCE"],
        "SHARED_SERVERLESS_COMPUTE": ["SERVERLESS"],
        "ALL_PURPOSE": ["ALL_PURPOSE"],
        "INTERACTIVE": ["ALL_PURPOSE"],
        "NOTEBOOKS": ["ALL_PURPOSE"],
    }.get(category, [])
    pool = [s for s in DBU_SKUS if any(p in s for p in patterns)]
    return random.choice(pool) if pool else random.choice(DBU_SKUS)

ITEM_CATEGORY_TO_BOP = {
    "job":              ["JOBS", "LAKEHOUSE_MONITORING", "PREDICTIVE_OPTIMIZATION"],
    "cluster":          ["ALL_PURPOSE", "INTERACTIVE", "NOTEBOOKS"],
    "warehouse":        ["SQL"],
    "endpoint":         ["MODEL_SERVING", "AI_GATEWAY", "AI_FUNCTIONS"],
    "pipeline":         ["DLT", "LAKEFLOW_CONNECT"],
    "vector_index":     ["VECTOR_SEARCH"],
    "notebook":         ["NOTEBOOKS", "INTERACTIVE"],
    "online_table":     ["ONLINE_TABLES"],
    "database_instance": ["DATABASE"],
}

USAGE_SCHEMA = spark.table(f"{BILLING}.usage").schema
USAGE_COLS = [f.name for f in USAGE_SCHEMA.fields]

# Identify decimal columns once so we can coerce float -> Decimal at row build time
DECIMAL_COLS = {f.name for f in USAGE_SCHEMA.fields if isinstance(f.dataType, DecimalType)}

# Struct fields need to be tuples in field-order, not dicts — PySpark Arrow conversion
# is positional.
def struct_field_names(name):
    dt = USAGE_SCHEMA[name].dataType
    return [f.name for f in dt.fields] if isinstance(dt, StructType) else None

META_FIELDS = struct_field_names("usage_metadata")

def dec(x):
    return Decimal(str(x))

def build_usage_row(d, item, bop):
    start_ts = datetime(d.year, d.month, d.day, random.randint(0, 23), 0, 0, tzinfo=timezone.utc)
    meta_values = {}
    for key in ("job_id", "cluster_id", "warehouse_id", "endpoint_id", "dlt_pipeline_id",
                "database_instance_id"):
        if key in item and (META_FIELDS and key in META_FIELDS):
            meta_values[key] = item[key]
    if "endpoint_name" in (META_FIELDS or ()) and "endpoint_id" in item:
        meta_values["endpoint_name"] = f"synth-ep-{item['endpoint_id'][:8]}"
    meta_tuple = tuple(meta_values.get(f) for f in META_FIELDS) if META_FIELDS else None

    row = {col: None for col in USAGE_COLS}
    row.update({
        "account_id": ACCOUNT_ID, "workspace_id": item["workspace_id"],
        "record_id": uuid.uuid4().hex, "sku_name": pick_sku(bop), "cloud": "AZURE",
        "usage_start_time": start_ts, "usage_end_time": start_ts + timedelta(hours=1),
        "usage_date": d, "custom_tags": item["tags"], "usage_unit": "DBU",
        "usage_quantity": dec(round(random.uniform(0.001, 5.0), 6)),
        "usage_metadata": meta_tuple, "identity_metadata": None,
        "record_type": "ORIGINAL", "ingestion_date": d,
        "billing_origin_product": bop, "product_features": None,
        "usage_type": "COMPUTE_TIME",
    })
    # Coerce any remaining float-valued decimal columns
    for c in DECIMAL_COLS:
        if isinstance(row[c], float):
            row[c] = dec(row[c])
    return tuple(row[c] for c in USAGE_COLS)

rows = []
for day_offset in range(DAYS):
    d = START_DATE + timedelta(days=day_offset)
    for ws in ALL_WORKSPACES:
        for item in ITEMS[ws["workspace_id"]]:
            bops = ITEM_CATEGORY_TO_BOP.get(item["category"], [])
            if not bops: continue
            for _ in range(random.randint(*OPS_PER_ITEM_PER_DAY)):
                rows.append(build_usage_row(d, item, random.choice(bops)))

# Item-less categories
for ws in ALL_WORKSPACES:
    dummy = {"workspace_id": ws["workspace_id"], "tags": gen_tags()}
    for day_offset in range(0, DAYS, 7):
        d = START_DATE + timedelta(days=day_offset)
        for bop in ("NETWORKING", "AGENT_EVALUATION", "SHARED_SERVERLESS_COMPUTE"):
            rows.append(build_usage_row(d, dummy, bop))

print(f"Synthesised {len(rows)} usage rows; writing to {BILLING}.usage...")
if rows:
    spark.createDataFrame(rows, schema=USAGE_SCHEMA).write.mode("append").saveAsTable(f"{BILLING}.usage")

## 7. Verify

Row counts per `billing_origin_product`; tag-coverage shape; SKU-name join sanity; estimated daily cost split between list and contracted.

In [ ]:
print("=== Rows per billing_origin_product ===")
spark.sql(f"""
    SELECT billing_origin_product, COUNT(*) AS rows,
           COUNT(DISTINCT workspace_id) AS workspaces,
           ROUND(SUM(usage_quantity), 4) AS total_dbus
    FROM {BILLING}.usage
    GROUP BY billing_origin_product
    ORDER BY rows DESC
""").show(50, truncate=False)

print("=== Service_ID tag coverage ===")
spark.sql(f"""
    SELECT CASE
            WHEN custom_tags['Service_ID'] IS NULL THEN 'untagged'
            WHEN size(split(custom_tags['Service_ID'], '/')) = 4
                 AND custom_tags['Service_ID'] NOT RLIKE '//'
                 AND custom_tags['Service_ID'] != '' THEN 'well_formed'
            ELSE 'malformed'
        END AS tag_state,
        COUNT(*) AS rows,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM {BILLING}.usage), 1) AS pct
    FROM {BILLING}.usage
    GROUP BY tag_state ORDER BY rows DESC
""").show(truncate=False)

print("=== SKU-name join sanity check ===")
spark.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN lp.sku_name IS NULL THEN 1 ELSE 0 END) AS unmatched_list,
           SUM(CASE WHEN ap.sku_name IS NULL THEN 1 ELSE 0 END) AS unmatched_account
    FROM {BILLING}.usage u
    LEFT JOIN {BILLING}.list_prices lp
        ON u.sku_name = lp.sku_name AND u.usage_start_time >= lp.price_start_time
        AND (lp.price_end_time IS NULL OR u.usage_start_time < lp.price_end_time)
    LEFT JOIN {BILLING}.account_prices ap
        ON u.sku_name = ap.sku_name AND u.usage_start_time >= ap.price_start_time
        AND (ap.price_end_time IS NULL OR u.usage_start_time < ap.price_end_time)
""").show(truncate=False)

print("=== Estimated cost by category ===")
spark.sql(f"""
    SELECT u.billing_origin_product,
           ROUND(SUM(u.usage_quantity * lp.pricing.default), 2) AS list_usd,
           ROUND(SUM(u.usage_quantity * ap.pricing.default), 2) AS contracted_usd
    FROM {BILLING}.usage u
    LEFT JOIN {BILLING}.list_prices lp
        ON u.sku_name = lp.sku_name AND u.usage_start_time >= lp.price_start_time
        AND (lp.price_end_time IS NULL OR u.usage_start_time < lp.price_end_time)
    LEFT JOIN {BILLING}.account_prices ap
        ON u.sku_name = ap.sku_name AND u.usage_start_time >= ap.price_start_time
        AND (ap.price_end_time IS NULL OR u.usage_start_time < ap.price_end_time)
    GROUP BY u.billing_origin_product ORDER BY list_usd DESC
""").show(50, truncate=False)

## 8. Run the cloud-infra-costs FOCUS query against the synthesised dataset

Loads the FOCUS query from GitHub, swaps `system.{schema}.` → `<DEV_CATALOG>.dev_{schema}.`, runs it, counts output rows. If it works against the synthesised data, it'll work against real `system.*` too.

In [ ]:
import urllib.request
focus_url = "https://raw.githubusercontent.com/databricks-solutions/cloud-infra-costs/main/focus/focus_query.sql"
focus_sql = urllib.request.urlopen(focus_url).read().decode()

# The cloud-infra-costs query has a deliberate placeholder for the user to inject
# their account-prices table path. Substitute ours.
focus_sql = focus_sql.replace("IDENTIFIER(:account_prices)", f"{BILLING}.account_prices")
for schema in ("billing", "compute", "access", "lakeflow"):
    focus_sql = focus_sql.replace(f"system.{schema}.", f"{DEV_CATALOG}.dev_{schema}.")

spark.sql(f"DROP TABLE IF EXISTS {BILLING}.focus_output")
spark.sql(f"CREATE TABLE {BILLING}.focus_output USING DELTA AS {focus_sql}")
n = spark.sql(f"SELECT COUNT(*) FROM {BILLING}.focus_output").collect()[0][0]
print(f"FOCUS query produced {n} rows; persisted to {BILLING}.focus_output")

print("\n=== Cost split by ServiceCategory ===")
spark.sql(f"""
    SELECT ServiceCategory, COUNT(*) AS rows,
           ROUND(SUM(BilledCost), 2) AS billed_usd,
           ROUND(SUM(ListCost), 2) AS list_usd,
           ROUND(SUM(ListCost) - SUM(EffectiveCost), 2) AS savings_usd
    FROM {BILLING}.focus_output
    GROUP BY 1 ORDER BY billed_usd DESC
""").show(20, truncate=False)